# PyTorch-Compatible Streaming Dataset Examples

This notebook demonstrates how to use the new streaming dataset classes for memory-efficient ML workflows.

## Features Demonstrated

1. **Streaming for Large Datasets** - Memory-efficient iteration
2. **Indexed Dataset with Shuffling** - Random access with caching
3. **Simple Batch Iterator** - Non-PyTorch batch processing
4. **Custom Transforms** - Data preprocessing and encoding
5. **Performance Comparison** - Memory usage and speed benchmarks
6. **Train/Test Split** - Creating separate datasets with WHERE clauses

In [1]:
# Import required libraries
import sys
import os
from pathlib import Path

# Add src to path if needed
project_root = Path.cwd().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import pbi
import pandas as pd
import numpy as np
from pbi import SequenceRetriever, PhageHostStreamingDataset, PhageHostIndexedDataset, phage_host_collate_fn

# Try to import PyTorch (optional)
try:
    import torch
    from torch.utils.data import DataLoader
    TORCH_AVAILABLE = True
    print("✅ PyTorch available")
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠️  PyTorch not available - some examples will be skipped")

print(f"PBI version: {pbi.__version__}")

✅ PyTorch available
PBI version: 0.1.0


In [2]:
# Connect to database using quick_connect
try:
    retriever = pbi.quick_connect()
    print("✅ Connected to database")
    
    # Get basic stats
    stats = retriever.get_stats()
    print("\n📊 Database Statistics:")
    print(f"  Phages: {stats['database']['phages']:,}")
    if 'hosts' in stats['database']:
        print(f"  Hosts: {stats['database']['hosts']:,}")
        print(f"  Phage-Host Associations: {stats['database'].get('phage_host_associations', 0):,}")
except Exception as e:
    print(f"❌ Error connecting: {e}")
    print("\n⚠️  Make sure you have run the workflow to create the database and FASTA files")
    raise

2026-02-18 07:11:06,881 - INFO - 📂 Checking FASTA index files:
2026-02-18 07:11:06,882 - INFO -    Phage index: True (52570.4 KB)
2026-02-18 07:11:06,884 - INFO -    Protein index: True (1432185.2 KB)
2026-02-18 07:11:06,885 - INFO - 📂 Using host mapping file: /data/processed/sequences/host_fasta_mapping.json
2026-02-18 07:11:06,889 - INFO -    Loaded mapping for 5354 hosts
2026-02-18 07:11:06,890 - INFO - 📂 Connecting to database: /data/processed/databases/phage_database_optimized.duckdb
2026-02-18 07:11:06,922 - INFO - 🔄 Starting background FASTA loading...
2026-02-18 07:11:06,924 - INFO - 🔄 [Background] Loading phage FASTA: /data/processed/sequences/all_phages.fasta
2026-02-18 07:11:06,925 - INFO - ✅ Initialization complete (FASTA loading in background)
2026-02-18 07:11:06,926 - INFO - ⏳ Waiting for FASTA loading to complete...


✅ Connected to database


2026-02-18 07:11:24,014 - INFO -    ✅ Phage FASTA loaded in 17.09s (873,717 sequences)
2026-02-18 07:11:24,016 - INFO - 🔄 [Background] Loading protein FASTA: /data/processed/sequences/all_proteins.fasta
2026-02-18 07:16:06,928 - WARNING - ⚠️ Timeout after 300s - FASTA may still be loading
2026-02-18 07:17:51,507 - INFO -    ✅ Protein FASTA loaded in 387.49s (31,050,116 sequences)
2026-02-18 07:17:51,509 - INFO -    ℹ️  Using on-demand loading for 5,354 individual host files
2026-02-18 07:17:51,510 - INFO - 🎉 All FASTA files loaded in 404.59s
2026-02-18 07:17:59,660 - INFO - 🔍 Sample phage keys:
2026-02-18 07:17:59,663 - INFO -    - 'AE002163.1...'
2026-02-18 07:17:59,664 - INFO -    - 'AF009630.1...'
2026-02-18 07:17:59,665 - INFO -    - 'AF011378.1...'
2026-02-18 07:17:59,667 - INFO - 🔍 Sample protein keys:
2026-02-18 07:17:59,667 - INFO -    - 'AE002163.1 AAF39720.1...'
2026-02-18 07:17:59,668 - INFO -    - 'AE002163.1 AAF39721.1...'
2026-02-18 07:17:59,669 - INFO -    - 'AE002163.1 


📊 Database Statistics:
  Phages: 873,718
  Hosts: 6,031
  Phage-Host Associations: 764,172


## Example 1: Streaming Dataset for Large Datasets

Use `PhageHostStreamingDataset` when:
- Working with very large datasets
- Memory is limited
- You don't need random access or shuffling
- Sequential processing is sufficient

**Note:** This example uses no WHERE clause to retrieve all available data. 
If you get an empty dataset, the streaming dataset will automatically:
- Check if phage-host associations exist in the database
- Show you what Completeness and Assembly_Level values are actually available
- Help you write filters that match your actual data

Add filters like `where_clause="LIMIT 1000"` or after checking available values:
`where_clause="p.Completeness = 'Complete'"` (use actual case from diagnostics)


In [4]:
print("=" * 80)
print("Example 1: Streaming Dataset for Large Datasets")
print("=" * 80)

# Create streaming dataset with filtering
# This only loads metadata for the query - sequences are fetched on-demand
# You can optionally track missing hosts with missing_hosts_csv="/path/to/output.csv"
streaming_dataset = retriever.create_streaming_dataset(
    where_clause=None,  # No filter - get all phage-host pairs
    batch_size=1000  # Fetch 1000 records at a time from database
)

print("\n✅ Created streaming dataset")
print("   Database batch size: 1000 records")
print("   Filter: None (all phage-host pairs)")

if TORCH_AVAILABLE:
    # Use with PyTorch DataLoader
    dataloader = DataLoader(
        streaming_dataset,
        batch_size=32,  # Model batch size
        num_workers=0,  # Use 0 for single worker (required for IterableDataset in notebooks),
        collate_fn=phage_host_collate_fn
    )
    
    print("\n📦 Iterating through batches...")
    
    # Process first few batches as demonstration
    max_batches = 5
    for i, batch in enumerate(dataloader):
        print(f" -- Debug DL i {i} -- ")
        if i >= max_batches:
            break
        
        # Batch is a dictionary with lists for each key
        print(f"\n  Batch {i+1}:")
        print(f"    Size: {len(batch['Phage_ID'])} samples")
        print(f"    Phage IDs (first 3): {batch['Phage_ID'][:3]}")
        print(f"    First phage sequence length: {len(batch['Phage_Sequence'][0]) if len(batch['Phage_Sequence']) > 0 else 0}")
        print(f"    First host sequence length: {len(batch['Host_Sequence'][0]) if len(batch['Host_Sequence']) > 0 else 0}")
        
        # Here you would typically:
        # - Encode sequences
        # - Convert to tensors
        # - Train your model
    
    print(f"\n✅ Processed {max_batches} batches successfully")
else:
    # Manual iteration without PyTorch
    print("\n📦 Iterating manually (without PyTorch)...")
    
    max_samples = 10
    for i, sample in enumerate(streaming_dataset):

        print(f" -- Debug i {i} -- ")
        if i >= max_samples:
            break
        
        print(f"\n  Sample {i+1}:")
        print(f"    Phage ID: {sample['Phage_ID']}")
        print(f"    Host ID: {sample['Host_ID']}")
        print(f"    Phage length: {len(sample['Phage_Sequence'])}")
        print(f"    Host length: {len(sample['Host_Sequence'])}")
    
    print(f"\n✅ Processed {max_samples} samples successfully")

2026-02-18 07:22:42,787 - INFO - Using host mapping with 5354 hosts


Example 1: Streaming Dataset for Large Datasets

✅ Created streaming dataset
   Database batch size: 1000 records
   Filter: None (all phage-host pairs)

📦 Iterating through batches...


2026-02-18 07:23:14,473 - WARNING - ⚠️  Host ID 'GCF_054696555_1' not found in mapping
2026-02-18 07:23:14,609 - WARNING - ⚠️  Host ID 'GCF_049723855_1' not found in mapping
2026-02-18 07:23:14,643 - WARNING - ⚠️  Host ID 'GCF_977870645_1' not found in mapping
2026-02-18 07:23:14,674 - WARNING - ⚠️  Host ID 'GCF_054790915_1' not found in mapping
2026-02-18 07:23:14,722 - WARNING - ⚠️  Host ID 'GCF_052076125_1' not found in mapping
2026-02-18 07:23:14,724 - WARNING - ⚠️  Host ID 'GCF_049723855_1' not found in mapping
2026-02-18 07:23:14,841 - WARNING - ⚠️  Host ID 'GCF_054130425_1' not found in mapping
2026-02-18 07:23:14,883 - WARNING - ⚠️  Host ID 'GCF_054130425_1' not found in mapping
2026-02-18 07:23:14,885 - WARNING - ⚠️  Host ID 'GCF_054790915_1' not found in mapping
2026-02-18 07:23:15,011 - WARNING - ⚠️  Host ID 'GCF_054790915_1' not found in mapping
2026-02-18 07:23:15,172 - WARNING - ⚠️  Host ID 'GCF_054696555_1' not found in mapping
2026-02-18 07:23:15,257 - WARNING - ⚠️  Hos

 -- Debug DL i 0 -- 

  Batch 1:
    Size: 32 samples
    Phage IDs (first 3): ['Station30_DCM_ALL_assembly_NODE_456_length_29851_cov_3.096792', 'Station30_DCM_ALL_assembly_NODE_1451_length_14269_cov_2.915154', 'Station32_SUR_ALL_assembly_NODE_402_length_29371_cov_3.749966']
    First phage sequence length: 29851
    First host sequence length: 4923101


2026-02-18 07:23:15,287 - WARNING - ⚠️  Host ID 'GCF_054392195_1' not found in mapping
2026-02-18 07:23:15,632 - WARNING - ⚠️  Host ID 'GCF_054790915_1' not found in mapping
2026-02-18 07:23:15,645 - WARNING - ⚠️  Host ID 'GCF_054693465_1' not found in mapping
2026-02-18 07:23:15,655 - WARNING - ⚠️  Host ID 'GCF_054696555_1' not found in mapping
2026-02-18 07:23:15,869 - WARNING - ⚠️  Host ID 'GCF_054790915_1' not found in mapping
2026-02-18 07:23:15,889 - WARNING - ⚠️  Host ID 'GCF_053175725_1' not found in mapping
2026-02-18 07:23:16,017 - WARNING - ⚠️  Host ID 'GCF_977870645_1' not found in mapping
2026-02-18 07:23:16,019 - WARNING - ⚠️  Host ID 'GCF_054696555_1' not found in mapping
2026-02-18 07:23:16,055 - WARNING - ⚠️  Host ID 'GCF_052956585_1' not found in mapping
2026-02-18 07:23:16,056 - WARNING - ⚠️  Host ID 'GCF_054693465_1' not found in mapping
2026-02-18 07:23:16,058 - WARNING - ⚠️  Host ID 'GCF_054790915_1' not found in mapping
2026-02-18 07:23:16,058 - WARNING - ⚠️  Hos

...

 -- Debug DL i 3 -- 

  Batch 4:
    Size: 32 samples
    Phage IDs (first 3): ['Station52_DCM_ALL_assembly_NODE_1901_length_10504_cov_2.448368', 'Station39_MES_COMBINED_FINAL_NODE_942_length_37465_cov_5.586501', 'Station39_MES_COMBINED_FINAL_NODE_2933_length_17070_cov_3.992830']
    First phage sequence length: 10504
    First host sequence length: 4122960


2026-02-18 07:23:17,053 - WARNING - ⚠️  Host ID 'GCF_054696555_1' not found in mapping
2026-02-18 07:23:17,234 - WARNING - ⚠️  Host ID 'GCF_052076125_1' not found in mapping
2026-02-18 07:23:17,235 - WARNING - ⚠️  Host ID 'GCF_052956585_1' not found in mapping
2026-02-18 07:23:17,236 - WARNING - ⚠️  Host ID 'GCF_052956585_1' not found in mapping
2026-02-18 07:23:17,270 - WARNING - ⚠️  Host ID 'GCF_052956585_1' not found in mapping
2026-02-18 07:23:17,302 - WARNING - ⚠️  Host ID 'GCF_054790915_1' not found in mapping
2026-02-18 07:23:17,326 - WARNING - ⚠️  Host ID 'GCF_052956585_1' not found in mapping
2026-02-18 07:23:17,332 - WARNING - ⚠️  Host ID 'GCF_054790915_1' not found in mapping
2026-02-18 07:23:17,337 - WARNING - ⚠️  Host ID 'GCF_054130425_1' not found in mapping
2026-02-18 07:23:17,348 - WARNING - ⚠️  Host ID 'GCF_054790915_1' not found in mapping
2026-02-18 07:23:17,354 - WARNING - ⚠️  Host ID 'GCF_052289055_1' not found in mapping


 -- Debug DL i 4 -- 

  Batch 5:
    Size: 32 samples
    Phage IDs (first 3): ['Station64_MES_COMBINED_FINAL_NODE_6371_length_10801_cov_3.671785', 'Station65_DCM_ALL_assembly_NODE_3013_length_10402_cov_3.073838', 'Station67_SUR_ALL_assembly_NODE_711_length_27748_cov_7.072907']
    First phage sequence length: 10801
    First host sequence length: 4923101


2026-02-18 07:23:17,408 - WARNING - ⚠️  Host ID 'GCF_054790915_1' not found in mapping
2026-02-18 07:23:17,410 - WARNING - ⚠️  Host ID 'GCF_052076125_1' not found in mapping
2026-02-18 07:23:17,428 - WARNING - ⚠️  Host ID 'GCF_052956585_1' not found in mapping
2026-02-18 07:23:17,443 - WARNING - ⚠️  Host ID 'GCF_054693465_1' not found in mapping
2026-02-18 07:23:17,460 - WARNING - ⚠️  Host ID 'GCF_054696555_1' not found in mapping
2026-02-18 07:23:17,461 - WARNING - ⚠️  Host ID 'GCF_054790915_1' not found in mapping
2026-02-18 07:23:17,474 - WARNING - ⚠️  Host ID 'GCF_054693465_1' not found in mapping


 -- Debug DL i 5 -- 

✅ Processed 5 batches successfully


## Example 2: Indexed Dataset with Shuffling

Use `PhageHostIndexedDataset` when:
- Dataset fits in memory (metadata cached)
- You need shuffling for better training
- You want multi-worker data loading
- Random access is needed

**Diagnostic Features:**
If the dataset is empty, you'll see detailed diagnostics showing:
- Total phage-host associations in database
- Available Completeness values (e.g., 'Complete', 'Partial', etc.)
- Available Assembly_Level values (e.g., 'Complete Genome', 'Scaffold', etc.)
- Whether the issue is with filters or missing sequence files

This helps you understand what data exists and write appropriate WHERE clauses.


In [ ]:
print("=" * 80)
print("Example 2: Indexed Dataset with Shuffling")
print("=" * 80)

# Create indexed dataset
# Metadata is cached in memory, sequences fetched on-demand
indexed_dataset = retriever.create_indexed_dataset(
    where_clause=None  # No filter - test with all data first
)

print(f"\n✅ Created indexed dataset")
print(f"   Total samples: {len(indexed_dataset):,}")
print("   Filter: None (all data - for testing)")

if len(indexed_dataset) == 0:
    print("\n⚠️  WARNING: Dataset is empty!")
    print("   This may be because:")
    print("   - The database has no data matching the filters")
    print("   - The Completeness values in the database differ from expected")
    print("   - No phage-host associations exist in the database")
    print("\n   Skipping this example...")
elif TORCH_AVAILABLE:
    # Use with PyTorch DataLoader - supports shuffling!
    dataloader = DataLoader(
        indexed_dataset,
        batch_size=32,
        shuffle=True,  # Enable shuffling for better training
        num_workers=0,  # Can use >0 in production (not in notebooks),
        collate_fn=phage_host_collate_fn
    )
    
    print("\n📦 Iterating with shuffling enabled...")
    
    # Process first few batches
    max_batches = 3
    for i, batch in enumerate(dataloader):
        if i >= max_batches:
            break
        
        print(f"\n  Batch {i+1}:")
        print(f"    Size: {len(batch['Phage_ID'])} samples")
        print(f"    Phage IDs: {batch['Phage_ID'][:3]}")
        print(f"    Host Species: {batch['Species_Name'][:3]}")
    
    print(f"\n✅ Shuffled iteration successful")
else:
    # Direct access by index (without PyTorch)
    print("\n📦 Random access examples (without PyTorch)...")
    
    if len(indexed_dataset) == 0:
        print("\n⚠️  Dataset is empty - no samples to show")
        print("   This may be due to data not being available or filters being too restrictive")
    else:
    
        # Access specific indices
        indices = [0, len(indexed_dataset)//2, len(indexed_dataset)-1]
        
        for idx in indices:
            sample = indexed_dataset[idx]
            print(f"\n  Sample at index {idx}:")
            print(f"    Phage ID: {sample['Phage_ID']}")
            print(f"    Host Species: {sample['Species_Name']}")
    
        print("\n✅ Random access successful")


## Example 3: Simple Batch Iterator

Use `get_phage_host_pairs_iterator()` when:
- You don't need PyTorch
- You want DataFrame batches
- Simple batch processing is sufficient
- Working with pandas-based workflows

In [5]:
print("=" * 80)
print("Example 3: Simple Batch Iterator (Non-PyTorch)")
print("=" * 80)

# Create iterator that yields DataFrame batches
batch_iterator = retriever.get_phage_host_pairs_iterator(
    where_clause="p.Length > 10000",  # Filter for larger phages
    batch_size=500
)

print("\n✅ Created batch iterator")
print("   Batch size: 500 pairs per batch")
print("   Filter: Phages > 10kb")

# Process batches
print("\n📦 Processing batches...")

max_batches = 3
all_gc_content = []

for i, batch_df in enumerate(batch_iterator):
    if i >= max_batches:
        break
    
    print(f"\n  Batch {i+1}:")
    print(f"    Rows: {len(batch_df)}")
    print(f"    Columns: {list(batch_df.columns)}")
    
    # Example analysis: compute GC content statistics
    avg_phage_gc = batch_df['Phage_GC'].mean()
    avg_host_gc = batch_df['Host_GC'].mean()
    
    print(f"    Avg Phage GC: {avg_phage_gc:.2f}%")
    print(f"    Avg Host GC: {avg_host_gc:.2f}%")
    
    all_gc_content.extend(batch_df['Phage_GC'].tolist())
    
    # You could export each batch to a file:
    # batch_df.to_csv(f'batch_{i}.csv', index=False)

print(f"\n✅ Processed {max_batches} batches")
print(f"   Total GC content values collected: {len(all_gc_content)}")

2026-02-18 07:23:56,762 - INFO - 🔍 Starting batch iteration with batch_size=500


Example 3: Simple Batch Iterator (Non-PyTorch)

✅ Created batch iterator
   Batch size: 500 pairs per batch
   Filter: Phages > 10kb

📦 Processing batches...


2026-02-18 07:24:00,244 - INFO - 📦 Processing batch 1 (987534 pairs)
2026-02-18 07:24:02,958 - WARNING - ⚠️  Phage sequence not found for ID: BK000583.1
2026-02-18 07:25:43,012 - WARNING - ⚠️  Host sequence not found for ID: GCF_054696555_1
2026-02-18 07:25:43,134 - WARNING - ⚠️  Host sequence not found for ID: GCF_049723855_1
2026-02-18 07:25:43,150 - WARNING - ⚠️  Host sequence not found for ID: GCF_977870645_1
2026-02-18 07:25:43,163 - WARNING - ⚠️  Host sequence not found for ID: GCF_054790915_1
2026-02-18 07:25:43,179 - WARNING - ⚠️  Host sequence not found for ID: GCF_052076125_1
2026-02-18 07:25:43,192 - WARNING - ⚠️  Host sequence not found for ID: GCF_054130425_1
2026-02-18 07:25:43,236 - WARNING - ⚠️  Host sequence not found for ID: GCF_054392195_1
2026-02-18 07:25:43,252 - WARNING - ⚠️  Host sequence not found for ID: GCF_054693465_1
2026-02-18 07:25:43,319 - WARNING - ⚠️  Host sequence not found for ID: GCF_053175725_1
2026-02-18 07:25:43,326 - WARNING - ⚠️  Host sequence n


  Batch 1:
    Rows: 987534
    Columns: ['Phage_ID', 'Host_ID', 'Phage_Source', 'Phage_Length', 'Phage_GC', 'Phage_Taxonomy', 'Phage_Completeness', 'Phage_Lifestyle', 'Phage_Cluster', 'Phage_Subcluster', 'Species_Name', 'Host_Assembly_Level', 'Host_Length', 'Host_GC', 'Host_RefSeq_Category', 'Phage_Sequence', 'Host_Sequence']
    Avg Phage GC: 44.77%
    Avg Host GC: 44.78%


2026-02-18 07:26:24,931 - INFO - 📦 Processing batch 2 (724978 pairs)
2026-02-18 07:27:47,053 - WARNING - ⚠️  Host sequence not found for ID: GCF_052161735_1
2026-02-18 07:27:47,308 - WARNING - ⚠️  Host sequence not found for ID: GCF_049723855_1
2026-02-18 07:27:47,309 - WARNING - ⚠️  Host sequence not found for ID: GCF_054693465_1
2026-02-18 07:27:47,310 - WARNING - ⚠️  Host sequence not found for ID: GCF_977056125_1
2026-02-18 07:27:47,344 - WARNING - ⚠️  Host sequence not found for ID: GCF_977056925_1
2026-02-18 07:27:47,353 - WARNING - ⚠️  Host sequence not found for ID: GCF_051057325_1
2026-02-18 07:27:47,365 - WARNING - ⚠️  Host sequence not found for ID: GCF_054696555_1
2026-02-18 07:27:47,395 - WARNING - ⚠️  Host sequence not found for ID: GCF_052346965_1
2026-02-18 07:27:47,465 - WARNING - ⚠️  Host sequence not found for ID: GCF_052076125_1
2026-02-18 07:27:47,466 - WARNING - ⚠️  Host sequence not found for ID: GCF_052975065_1
2026-02-18 07:27:47,471 - WARNING - ⚠️  Host sequen


  Batch 2:
    Rows: 724978
    Columns: ['Phage_ID', 'Host_ID', 'Phage_Source', 'Phage_Length', 'Phage_GC', 'Phage_Taxonomy', 'Phage_Completeness', 'Phage_Lifestyle', 'Phage_Cluster', 'Phage_Subcluster', 'Species_Name', 'Host_Assembly_Level', 'Host_Length', 'Host_GC', 'Host_RefSeq_Category', 'Phage_Sequence', 'Host_Sequence']
    Avg Phage GC: 44.80%
    Avg Host GC: 44.81%

✅ Processed 3 batches
   Total GC content values collected: 1712512


## Example 4: Custom Transforms

Apply custom transformations to data:
- Sequence encoding (one-hot, k-mers)
- Feature extraction
- Data augmentation
- Normalization

In [6]:
print("=" * 80)
print("Example 4: Custom Transforms")
print("=" * 80)

# Define custom transform function
def sequence_encoding_transform(sample):
    """
    Transform that adds encoded features to the sample.
    
    This example computes:
    - Sequence length features
    - GC content features
    - Simple nucleotide counts
    """
    phage_seq = sample['Phage_Sequence']
    host_seq = sample['Host_Sequence']
    
    # Length features
    sample['phage_length'] = len(phage_seq)
    sample['host_length'] = len(host_seq)
    sample['length_ratio'] = len(phage_seq) / max(len(host_seq), 1)
    
    # Nucleotide composition for phage
    phage_upper = phage_seq.upper()
    sample['phage_gc_computed'] = ((phage_upper.count('G') + phage_upper.count('C')) / 
                                    max(len(phage_seq), 1)) * 100
    sample['phage_a_count'] = phage_upper.count('A')
    sample['phage_t_count'] = phage_upper.count('T')
    sample['phage_g_count'] = phage_upper.count('G')
    sample['phage_c_count'] = phage_upper.count('C')
    
    # Optionally keep or remove raw sequences to save memory
    # del sample['Phage_Sequence']
    # del sample['Host_Sequence']
    
    return sample

# Create dataset with transform
transformed_dataset = retriever.create_indexed_dataset(
    where_clause="LIMIT 100",
    transform=sequence_encoding_transform
)

print(f"\n✅ Created dataset with custom transform")
print(f"   Total samples: {len(transformed_dataset)}")

if len(transformed_dataset) == 0:
    print("\n⚠️  Dataset is empty - cannot show sample")
    print("   This may be due to data not being available or filters being too restrictive")
else:

    # Get a sample to see the transformed data
    sample = transformed_dataset[0]
    print("\n📊 Transformed sample fields:")
    for key in sorted(sample.keys()):
        if key not in ['Phage_Sequence', 'Host_Sequence']:  # Skip long sequences in print
            print(f"   {key}: {sample[key]}")

    print("\n✅ Transform applied successfully")

Example 4: Custom Transforms


2026-02-18 07:28:19,794 - INFO - Using host mapping with 5354 hosts


ParserException: Parser Error: syntax error at or near "LIMIT"

LINE 21:          WHERE LIMIT 100
                        ^

## Example 5: Performance Comparison

Compare different approaches:
- Loading all at once vs streaming
- Memory usage
- Iteration speed

In [7]:
print("=" * 80)
print("Example 5: Performance Comparison")
print("=" * 80)

import time
import gc

# Test with a limited dataset for comparison
test_where = "LIMIT 500"

print("\n🔬 Test Configuration:")
print(f"   Filter: {test_where}")

# Method 1: Load all at once (traditional approach)
print("\n📊 Method 1: Load All At Once")
gc.collect()
start_time = time.time()

all_at_once_df = retriever.get_phage_host_pairs(where_clause=test_where.replace(' LIMIT 500', ''))
all_at_once_df = all_at_once_df.head(500)  # Limit to 500 for fair comparison

load_all_time = time.time() - start_time
memory_mb = all_at_once_df.memory_usage(deep=True).sum() / 1024 / 1024

print(f"   Time: {load_all_time:.2f}s")
print(f"   DataFrame memory: {memory_mb:.2f} MB")
print(f"   Rows: {len(all_at_once_df)}")

# Method 2: Streaming dataset
print("\n📊 Method 2: Streaming Dataset")
gc.collect()
start_time = time.time()

streaming_ds = retriever.create_streaming_dataset(
    where_clause=test_where.replace(' LIMIT 500', ''),
    batch_size=100
)

count = 0
for i, sample in enumerate(streaming_ds):
    count += 1
    if count >= 500:
        break

streaming_time = time.time() - start_time

print(f"   Time: {streaming_time:.2f}s")
print(f"   Samples processed: {count}")
print(f"   Note: Streaming uses minimal memory (no full DataFrame)")

# Method 3: Batch iterator
print("\n📊 Method 3: Batch Iterator")
gc.collect()
start_time = time.time()

count = 0
for batch_df in retriever.get_phage_host_pairs_iterator(
    where_clause=test_where.replace(' LIMIT 500', ''),
    batch_size=100
):
    count += len(batch_df)
    if count >= 500:
        break

iterator_time = time.time() - start_time

print(f"   Time: {iterator_time:.2f}s")
print(f"   Samples processed: {count}")
print(f"   Note: Processes one batch at a time")

# Summary
print("\n📈 Performance Summary:")
print(f"   Load all at once: {load_all_time:.2f}s ({memory_mb:.2f} MB)")
print(f"   Streaming:        {streaming_time:.2f}s (minimal memory)")
print(f"   Batch iterator:   {iterator_time:.2f}s (batch-sized memory)")
print("\n💡 Recommendations:")
print("   - Use streaming for very large datasets (100k+ pairs)")
print("   - Use indexed dataset for medium datasets with shuffling")
print("   - Use batch iterator for pandas-based workflows")

2026-02-18 07:48:53,760 - INFO - 🔍 Querying phage-host pairs...


Example 5: Performance Comparison

🔬 Test Configuration:
   Filter: LIMIT 500

📊 Method 1: Load All At Once


ParserException: Parser Error: syntax error at or near "LIMIT"

LINE 21:          WHERE LIMIT 500
                        ^

## Example 6: Train/Test Split with Streaming

Create separate datasets for training and testing using WHERE clauses.
This avoids data leakage and allows efficient memory usage.

In [9]:
print("=" * 80)
print("Example 6: Train/Test Split with Streaming")
print("=" * 80)

# Strategy: Split data using different quality thresholds
# Train on high-quality data, test on medium-quality data

# Training dataset: Complete phages with complete host genomes
train_dataset = retriever.create_streaming_dataset(
    where_clause="LIMIT 5000",  # Limit for reasonable performance
    batch_size=1000
)

print("\n✅ Created training dataset")
print("   Filter: First 5000 pairs (for reasonable performance)")
print("   Type: Streaming (for large-scale training)")

# Test dataset: Complete phages with scaffold-level assemblies
test_dataset = retriever.create_indexed_dataset(
    where_clause="LIMIT 1000 OFFSET 5000"  # Different subset for testing
)

print("\n✅ Created test dataset")
print(f"   Filter: Next 1000 pairs (offset from training set)")
print(f"   Size: {len(test_dataset)} samples")
print("   Type: Indexed (for easier evaluation)")

if TORCH_AVAILABLE:
    # Create DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        num_workers=0,
    collate_fn=phage_host_collate_fn,
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,  # No shuffling for test set
        num_workers=0,
    collate_fn=phage_host_collate_fn,
    )
    
    print("\n📦 DataLoaders created")
    print("   Train: Streaming, no shuffle")
    print("   Test: Indexed, no shuffle")
    
    # Simulate training loop
    print("\n🔄 Simulating training loop...")
    
    # Process a few training batches
    print("\n  Training batches:")
    for i, batch in enumerate(train_loader):
        if i >= 3:
            break
        print(f"    Batch {i+1}: {len(batch['Phage_ID'])} samples")
        # Here: model.train(), forward pass, compute loss, backward pass, optimize
    
    # Process a few test batches
    print("\n  Test batches:")
    for i, batch in enumerate(test_loader):
        if i >= 3:
            break
        print(f"    Batch {i+1}: {len(batch['Phage_ID'])} samples")
        # Here: model.eval(), forward pass, compute metrics
    
    print("\n✅ Train/test split demonstration complete")
else:
    print("\n⚠️  PyTorch not available - skipping DataLoader demonstration")

print("\n💡 Alternative splitting strategies:")
print("   - By source: WHERE p.Source_DB = 'PhagesDB' (train) vs 'INPHARED' (test)")
print("   - By taxonomy: Different phage families for train/test")
print("   - Random split: Use modulo on Phage_ID hash for deterministic split")
print("   - Temporal: WHERE h.Download_Date < '2023-01-01' (train) vs >= (test)")

2026-02-18 07:49:57,363 - INFO - Using host mapping with 5354 hosts
2026-02-18 07:49:57,375 - INFO - Using host mapping with 5354 hosts


Example 6: Train/Test Split with Streaming

✅ Created training dataset
   Filter: First 5000 pairs (for reasonable performance)
   Type: Streaming (for large-scale training)


ParserException: Parser Error: syntax error at or near "LIMIT"

LINE 21:          WHERE LIMIT 1000 OFFSET 5000
                        ^

## Example 7: Tracking Missing Hosts

The datasets can automatically track phages whose host genomes could not be retrieved.
This is useful for:
- Understanding which hosts are missing from your FASTA collection
- Debugging pipeline issues
- Identifying which hosts need to be downloaded

The missing hosts CSV contains:
- **Phage_ID**: The phage that was affected
- **Host_ID**: The host genome that couldn't be retrieved
- **Species_Name**: The host species name
- **Phage metadata**: Source, Length, Taxonomy for context
- **Failure_Reason**: Specific reason (not in mapping, file not found, index error, etc.)


In [ ]:
print("=" * 80)
print("Example 7: Tracking Missing Hosts")
print("=" * 80)

# Create dataset with missing hosts tracking enabled
dataset_with_tracking = retriever.create_streaming_dataset(
    where_clause=None,  # Try to load all pairs
    batch_size=1000,
    missing_hosts_csv="/tmp/pbi_missing_hosts.csv"  # Save missing hosts here
)

print("\n✅ Created dataset with missing hosts tracking")
print("   Missing hosts will be saved to: /tmp/pbi_missing_hosts.csv")

# Iterate through a sample of the data
print("\n📦 Processing samples...")
sample_count = 0
max_samples = 100

for sample in dataset_with_tracking:
    sample_count += 1
    if sample_count >= max_samples:
        break

print(f"\n✅ Processed {sample_count} samples")

# Check if the CSV was created
import os
if os.path.exists("/tmp/pbi_missing_hosts.csv"):
    print("\n📄 Missing hosts CSV created!")
    
    # Read and display summary
    import pandas as pd
    missing_df = pd.read_csv("/tmp/pbi_missing_hosts.csv")
    
    print(f"   Total missing hosts: {len(missing_df)}")
    print(f"\n   Failure reasons breakdown:")
    print(missing_df['Failure_Reason'].value_counts().to_string())
    
    print(f"\n   Sample of missing hosts:")
    print(missing_df[['Phage_ID', 'Host_ID', 'Species_Name', 'Failure_Reason']].head(10).to_string())
else:
    print("\n✅ No missing hosts! All host genomes were successfully retrieved.")

print("\n💡 Use this CSV to:")
print("   - Identify which host genomes need to be downloaded")
print("   - Debug file system issues (missing files, corrupt indexes)")
print("   - Understand data completeness in your phage-host dataset")


## Summary and Best Practices

### When to Use Each Approach

1. **PhageHostStreamingDataset**
   - ✅ Very large datasets (millions of pairs)
   - ✅ Limited memory available
   - ✅ Sequential processing is sufficient
   - ❌ No shuffling support
   - ❌ No random access

2. **PhageHostIndexedDataset**
   - ✅ Medium-sized datasets (thousands to hundreds of thousands)
   - ✅ Need shuffling for better training
   - ✅ Multi-worker data loading
   - ✅ Random access needed
   - ❌ Metadata must fit in memory

3. **get_phage_host_pairs_iterator()**
   - ✅ Pandas-based workflows
   - ✅ Don't need PyTorch
   - ✅ Batch processing
   - ✅ Simple integration
   - ❌ No PyTorch integration

### Performance Tips

- **Batch size**: Larger batches = fewer database queries, but more memory
- **WHERE clauses**: Filter early at the database level for efficiency
- **Transforms**: Apply heavy preprocessing in transforms, not in training loop
- **Workers**: Use num_workers > 0 in production (not in Jupyter notebooks)
- **Caching**: Indexed dataset caches metadata; streaming does not

### Data Quality

- Always filter for complete/high-quality genomes when possible
- Check for missing sequences (handled gracefully with warnings)
- Validate data before heavy computation
- Use WHERE clauses to ensure data quality

### Memory Management

- Streaming: O(batch_size) memory usage
- Indexed: O(metadata_size + batch_size) memory usage
- Load all: O(full_dataset) memory usage

Choose appropriately based on your dataset size and available RAM.

In [ ]:
# Cleanup
print("\n🧹 Cleaning up...")
retriever.close()
print("✅ Done!")